In [8]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import time
import os
import glob

# 🔥 🔥 🔥 初始化全局变量
PARAMS = {}
alpha_values = {}


def select_alpha_range(target_daily_tasks):
    """根据目标日均任务数，选择合适的退化率范围"""
    if target_daily_tasks <= 5:  # 稀疏
        # return 0.004, 0.006
        return 0.001, 0.002
    elif 6 <= target_daily_tasks <= 12:  # 中等
        return 0.010, 0.013
    else:  # 13+ 密集
        return 0.020, 0.025


def generate_maintenance_plan(N, T, random_seed, target_daily_tasks):
    """生成维护计划（新版本：基于目标任务密度）"""
    random.seed(random_seed)
    
    caissons = [i for i in range(N)]
    initial_p_values = [round(random.uniform(3.2, 3.5), 2) for _ in caissons]
    
    a, b = select_alpha_range(target_daily_tasks)
    alpha_values = [round(random.uniform(a, b), 6) for _ in caissons]
    
    columns = ['INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE']
    data = []
    
    index = 0
    for caisson in range(N):
        current_p = initial_p_values[caisson]
        total_days = 0
        alpha = alpha_values[caisson]
        
        while total_days < T:
            time_to_3_2 = int((current_p - 3.2) / alpha)
            if total_days + time_to_3_2 >= T:
                break
            
            operation = 'x'
            data.append([index, caisson, initial_p_values[caisson], operation, total_days + time_to_3_2])
            index += 1
            total_days += time_to_3_2 + 1
            current_p = 3.5
    
    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan, alpha_values


def calculate_maintenance_period_from_data(df, d_min_ratio=0.6, d_max_ratio=1.4, window_size=10):
    """从数据中提取维修周期"""
    maintenance_periods = {}
    D_min_dict = {}
    D_max_dict = {}
    
    print("\n===== 从数据中提取维修周期 =====")
    print(f"{'Caisson':<10} {'间隔数':<10} {'间隔范围':<20} {'周期':<12} {'D_min':<10} {'D_max':<10}")
    print("-" * 80)
    
    for caisson_id in sorted(df['CSEM'].unique()):
        caisson_tasks = df[(df['CSEM'] == caisson_id) & (df['OPERATION'] == 'x')].sort_values('DATE')
        
        if len(caisson_tasks) < 2:
            default_period = window_size
            maintenance_periods[caisson_id] = default_period
            D_min_dict[caisson_id] = int(np.ceil(default_period * d_min_ratio))
            D_max_dict[caisson_id] = int(np.ceil(default_period * d_max_ratio))
            print(f"{caisson_id:<10} ... {default_period:<12.2f} ...")  
            continue
        
        intervals = []
        for i in range(1, len(caisson_tasks)):
            interval = caisson_tasks.iloc[i]['DATE'] - caisson_tasks.iloc[i-1]['DATE']
            intervals.append(interval)
        
        maintenance_period = np.mean(intervals)
        maintenance_periods[caisson_id] = maintenance_period
        D_min_dict[caisson_id] = int(np.ceil(maintenance_period * d_min_ratio))
        D_max_dict[caisson_id] = int(np.ceil(maintenance_period * d_max_ratio))
        
        interval_min = min(intervals)
        interval_max = max(intervals)
        interval_range = f"[{interval_min}, {interval_max}]"
        
        print(f"{caisson_id:<10} {len(intervals):<10} {interval_range:<20} "
              f"{maintenance_period:<12.2f} {D_min_dict[caisson_id]:<10} {D_max_dict[caisson_id]:<10}")
    
    print("===== 提取完成 =====\n")
    return maintenance_periods, D_min_dict, D_max_dict


def clustering_CR_simple(df_window, window_size, window_start, Cmax=5):
    """3天简单聚合策略"""
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window
    
    print(f"Window {window_start}: N_tasks={len(df_window)}")
    
    # 按原始时间排序
    df_window = df_window.sort_values('window_time').copy()
    
    # 初始化CENTER为原始时间
    df_window['CENTER'] = df_window['DATE']
    
    # 3天聚合逻辑
    days_3block = 3
    for day in range(window_start, window_start + window_size, days_3block):
        # 获取该3天块内的任务
        mask_3days = (df_window['window_time'] >= day - window_start) & \
                    (df_window['window_time'] < day + days_3block - window_start)
        tasks_3days = df_window[mask_3days].copy()
        
        if tasks_3days.empty:
            continue
            
        # 按时间排序，优先第1天聚合
        tasks_3days = tasks_3days.sort_values('window_time')
        
        # 第1天聚合（不超过Cmax）
        day1_time = day
        tasks_day1 = []
        for idx, row in tasks_3days.iterrows():
            if len(tasks_day1) < Cmax:
                tasks_day1.append(idx)
            else:
                break
        
        # 更新第1天任务
        df_window.loc[tasks_day1, 'CENTER'] = day1_time
        
        # 剩余任务→第2天
        remaining_tasks = tasks_3days.drop(tasks_day1).index.tolist()
        if remaining_tasks:
            day2_time = day + 1
            tasks_day2 = []
            for idx in remaining_tasks:
                if len(tasks_day2) < Cmax:
                    tasks_day2.append(idx)
                else:
                    break
            df_window.loc[tasks_day2, 'CENTER'] = day2_time
            remaining_tasks = [idx for idx in remaining_tasks if idx not in tasks_day2]
        
        # 最后剩余→第3天
        if remaining_tasks:
            day3_time = day + 2
            df_window.loc[remaining_tasks, 'CENTER'] = day3_time
    
    print(f"  → 聚合完成: {df_window['CENTER'].nunique()} 个聚类点")
    return df_window


def calculate_gas_leakage(df, T, alpha_values, K=5.993, use_center=True):
    """泄漏计算"""
    total_leakage = 0
    for csem in df['CSEM'].unique():
        csem_data = df[df['CSEM'] == csem]
        x_operations = csem_data[csem_data['OPERATION'] == 'x'].sort_values('DATE' if not use_center else 'CENTER')
        if x_operations.empty:
            continue
            
        alpha = alpha_values[int(csem)]

        # 任务间泄漏
        for i in range(1, len(x_operations)):
            current_time = x_operations.iloc[i][('CENTER' if use_center else 'DATE')]
            prev_time = x_operations.iloc[i-1][('CENTER' if use_center else 'DATE')]
            center_diff = current_time - prev_time - 1
            total_leakage += max(0, center_diff) * alpha * K

        # 开头泄漏
        first_time = x_operations.iloc[0][('CENTER' if use_center else 'DATE')]
        total_leakage += first_time * alpha * K

        # 结尾泄漏
        last_time = x_operations.iloc[-1][('CENTER' if use_center else 'DATE')]
        leak_time = max(0, T - (last_time + 1))
        total_leakage += leak_time * alpha * K
    return total_leakage


def update_secondary_tasks(main_df, clustered_df, window_start, window_end, alpha_values):
    """更新次要任务时间"""
    for caisson in clustered_df["CSEM"].unique():
        tasks = main_df[
            (main_df['CSEM'] == caisson) & 
            (main_df['DATE'].between(window_start, window_end)) & 
            (main_df['OPERATION'] == 'x')
        ].sort_values('DATE')
        
        if len(tasks) > 1:
            second_idx = tasks.index[1]
            center_time = clustered_df.loc[clustered_df['CSEM'] == caisson, 'CENTER'].iloc[0]
            alpha = alpha_values[int(caisson)]
            new_date = int(center_time + 1 + (3.5 - 3.2) / alpha)
            main_df.loc[second_idx, ['DATE', 'CENTER']] = new_date, new_date


def run_clustering_analysis(params=None):
    """完整聚类分析（简单策略版）"""
    global alpha_values, PARAMS

    # 🔥 🔥 🔥 关键修复：初始化全局 PARAMS
    if 'PARAMS' not in globals():
        PARAMS = {}
    
    # 使用默认参数或覆盖参数
    if params is not None:
        PARAMS.update(params)
    
    # 从参数中读取target_daily_tasks
    target_daily_tasks = PARAMS.get('target_daily_tasks', PARAMS['P'] / (PARAMS['T'] / 30))
    
    # 生成维护计划
    df, alpha_values_list = generate_maintenance_plan(
        N=PARAMS['P'], T=PARAMS['T'], random_seed=PARAMS.get('random_seed', 1), 
        target_daily_tasks=target_daily_tasks
    )
    alpha_values = {i: alpha_values_list[i] for i in range(PARAMS['P'])}
    
    print(f"Generated {len(df)} tasks (target_daily_tasks={target_daily_tasks:.1f})")
    
    # 计算维修周期
    maintenance_periods, D_min_dict, D_max_dict = calculate_maintenance_period_from_data(
        df, window_size=PARAMS['window_size']
    )
    
    # CPU时间测量
    clustering_cpu_start = time.process_time()
    
    df_init = df.copy()
    df_history_list = [df_init]
    sliding_windows_starts = []
    sliding_windows_ends = []
    df['CENTER'] = df['DATE']
    
    window_start = 0
    window_size = PARAMS['window_size']
    
    while window_start <= PARAMS['T']:
        window_end = min(window_start + window_size, PARAMS['T'])
        sliding_windows_starts.append(window_start)
        sliding_windows_ends.append(window_end)
        
        mask = (df['DATE'] >= window_start) & (df['DATE'] <= window_end) & (df['OPERATION']== 'x')
        window_df = df.loc[mask].copy()
        if window_df.empty:
            break
            
        window_df['window_time'] = window_df['DATE'] - window_start
        first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])

        clustered_df = clustering_CR_simple(first_tasks, window_size, window_start, PARAMS['Cmax'])
        
        if clustered_df is not None:
            center_map = clustered_df.set_index('INDEX')['CENTER']
            df.loc[df['INDEX'].isin(center_map.index), 'CENTER'] = df['INDEX'].map(center_map)
            update_secondary_tasks(df, clustered_df, window_start, window_end, alpha_values)
        
        # 滑动窗口逻辑
        second_dates = []
        if clustered_df is not None:
            for caisson in clustered_df["CSEM"].unique():
                tasks = df[(df['CSEM'] == caisson) & 
                          (df['OPERATION'] == 'x') &
                          (df['DATE'].between(window_start, window_end))].sort_values('DATE')
                if len(tasks) > 1:
                    second_dates.append(tasks.iloc[1]['DATE'])
        
        window_start = min(second_dates) if second_dates else window_start + window_size
        df_history_list.append(df.copy())
        
        if window_end >= PARAMS['T']:
            break
    
    # 处理未分配任务
    mask_remain = (df['OPERATION'] == "x") & (df['CENTER'] == df['DATE'])
    df.loc[mask_remain, 'CENTER'] = df.loc[mask_remain, 'DATE']

    clustering_cpu_end = time.process_time()
    total_clustering_cpu_time = clustering_cpu_end - clustering_cpu_start

    # 计算泄漏
    leakage_before = calculate_gas_leakage(df_init, PARAMS['T'], alpha_values, PARAMS['K'], use_center=False)
    leakage_after = calculate_gas_leakage(df, PARAMS['T'], alpha_values, PARAMS['K'], use_center=True)
    reduction = leakage_before - leakage_after
    reduction_pct = (reduction / leakage_before * 100) if leakage_before > 0 else 0
    
    return {
        'final_df': df,
        'df_history_list': df_history_list,
        'sliding_windows_starts': sliding_windows_starts,
        'sliding_windows_ends': sliding_windows_ends,
        'alpha_values': alpha_values_list,
        'maintenance_plan': df_init,
        'D_min_dict': D_min_dict,
        'D_max_dict': D_max_dict,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'reduction': reduction,
        'reduction_pct': reduction_pct,
        'total_clustering_cpu_time': total_clustering_cpu_time
    }


# 🔥 🔥 🔥 修复版：自动添加 N 和 target_daily_tasks 字段
def calculate_detailed_stats(result_dict, PARAMS, seed=None):
    """计算详细统计 - 修复版，自动添加N和target_daily_tasks"""
    df_orig = result_dict['maintenance_plan'].copy()
    result_df = result_dict['final_df'].copy()
    
    # 只针对x任务
    df_x = df_orig[df_orig['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    res_x = result_df[result_df['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    
    time_diffs = res_x['CENTER'].values - df_x['DATE'].values
    
    # CP成本
    cp_cost = np.sum(np.where(time_diffs < 0, np.abs(time_diffs) * PARAMS['Calpha'], 
                             time_diffs * PARAMS['Cbeta']))
    
    # 聚类统计
    center_counts = res_x['CENTER'].value_counts()
    num_groups = len(center_counts)
    avg_tasks_per_group = center_counts.mean()
    
    # 时间偏移
    num_tasks = len(time_diffs)
    avg_shift_abs = np.mean(np.abs(time_diffs))
    max_shift_abs = np.max(np.abs(time_diffs))
    
    # 提前/延迟统计（修复pct计算）
    advanced_mask = time_diffs < 0
    avg_advance = np.mean(np.abs(time_diffs[advanced_mask])) if np.any(advanced_mask) else 0
    num_advanced = np.sum(advanced_mask)
    pct_advanced = (num_advanced / num_tasks * 100) if num_tasks > 0 else 0
    
    delayed_mask = time_diffs > 0
    avg_delay = np.mean(time_diffs[delayed_mask]) if np.any(delayed_mask) else 0
    num_delayed = np.sum(delayed_mask)
    pct_delayed = (num_delayed / num_tasks * 100) if num_tasks > 0 else 0
    
    num_unchanged = np.sum(time_diffs == 0)
    pct_unchanged = (num_unchanged / num_tasks * 100) if num_tasks > 0 else 0
    
    # CS成本
    cs_cost_before = len(df_x) * PARAMS['Cs']
    cs_cost_after = num_groups * PARAMS['Cs']
    total_cost = cs_cost_after + cp_cost
    
    # 泄漏
    leakage_before = result_dict['leakage_before']
    leakage_after = result_dict['leakage_after']
    leakage_reduction = result_dict['reduction']
    leakage_reduction_pct = result_dict['reduction_pct']
    
    total_clustering_cpu_time = result_dict.get('total_clustering_cpu_time', 0)
    
    stats = {
        'seed': seed,
        'N': PARAMS['P'],  # 🔥 自动添加N
        'target_daily_tasks': PARAMS.get('target_daily_tasks', PARAMS['P'] / (PARAMS['T'] / 30)),  # 🔥 自动添加密度
        'nb_groups': num_groups,
        'Avg_tasks/group': round(avg_tasks_per_group, 3),
        'CS_cost_before': cs_cost_before,
        'CS_cost_after': cs_cost_after,
        'CP_cost': round(cp_cost, 3),
        'Total_cost': round(total_cost, 3),
        'Avg_shift_(abs)': round(avg_shift_abs, 3),
        'Max_shift_(abs)': round(max_shift_abs, 3),
        'Avg_advance_(days)': round(avg_advance, 3),
        'Avg_delay_(days)': round(avg_delay, 3),
        'pct_advanced_%': round(pct_advanced, 3),
        'pct_delayed_%': round(pct_delayed, 3),
        'pct_unchanged_%': round(pct_unchanged, 3),
        'leakage_before': round(leakage_before, 3),
        'leakage_after': round(leakage_after, 3),
        'leakage_reduction': round(leakage_reduction, 3),
        'leakage_reduction_pct': round(leakage_reduction_pct, 3),
        'total_clustering_cpu_s': round(total_clustering_cpu_time, 3)
    }
    return stats


# 🔥 🔥 🔥 修复版批量运行函数
def run_analysis_with_csv_export(PARAMS, seeds, output_filename='detailed_results.csv'):
    """批量运行 - 完全修复版，自动包含N和target_daily_tasks"""
    all_results = []
    
    for seed in seeds:
        print(f"     🌱 Seed {seed}...")
        
        params_copy = PARAMS.copy()
        params_copy['random_seed'] = seed
        
        result_dict = run_clustering_analysis(params_copy)
        stats = calculate_detailed_stats(result_dict, params_copy, seed=seed)
        all_results.append(stats)
    
    results_df = pd.DataFrame(all_results)
    
    # 🔥 精确列顺序（与MIMAR模型完全对齐）
    exact_columns = [
        'seed', 'N', 'target_daily_tasks', 'nb_groups', 'Avg_tasks/group', 'CS_cost_before', 'CS_cost_after',
        'CP_cost', 'Total_cost', 'Avg_shift_(abs)', 'Max_shift_(abs)',
        'Avg_advance_(days)', 'Avg_delay_(days)', 'pct_advanced_%', 'pct_delayed_%',
        'pct_unchanged_%', 'leakage_before', 'leakage_after', 'leakage_reduction',
        'leakage_reduction_pct', 'total_clustering_cpu_s'
    ]
    
    # 确保所有列存在
    for col in exact_columns:
        if col not in results_df.columns:
            results_df[col] = np.nan
    
    results_df = results_df[exact_columns]
    results_df.to_csv(output_filename, index=False, float_format='%.3f')
    
    return results_df


# 🔥 🔥 🔥 N敏感性分析主函数（适配简单策略）
def run_N_sensitivity_analysis_simple(base_params, task_densities, N_values, seeds, output_dir='N_sensitivity_Simple_results'):
    """固定T，对每个任务密度做N敏感性分析 - 简单策略版"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    all_results = []
    
    for target_daily_tasks in task_densities:
        print(f"\n{'='*60}")
        print(f"🔍 Simple Strategy - 任务密度: {target_daily_tasks} tasks/day")
        print(f"{'='*60}")
        
        density_results = []
        
        for N in N_values:
            print(f"\n  📊 Testing N={N}...")
            
            params_template = base_params.copy()
            params_template['P'] = N
            params_template['target_daily_tasks'] = target_daily_tasks
            
            filename = f"{output_dir}/density_{target_daily_tasks}_N_{N}_all_seeds.csv"
            
            result_df = run_analysis_with_csv_export(
                params_template, seeds, output_filename=filename
            )
            
            density_results.append(result_df)
            print(f"     ✅ N={N} results saved: {filename}")
        
        # 每个密度汇总所有N
        density_summary = pd.concat(density_results, ignore_index=True)
        density_summary_filename = f"{output_dir}/SUMMARY_density_{target_daily_tasks}_all_N.csv"
        density_summary.to_csv(density_summary_filename, index=False)
        print(f"  📁 Density {target_daily_tasks} summary: {density_summary_filename}")
        
        all_results.append(density_summary)
    
    # 最终大汇总
    grand_summary = pd.concat(all_results, ignore_index=True)
    grand_summary_filename = f"{output_dir}/GRAND_SUMMARY_all_densities_all_N.csv"
    grand_summary.to_csv(grand_summary_filename, index=False)
    print(f"\n🎉 Simple Strategy总汇总保存: {grand_summary_filename}")
    
    return grand_summary


############################################################
# 🔥 🔥 🔥 主实验配置
############################################################

if __name__ == "__main__":
    
    # 🔥 固定T和其他参数
    BASE_PARAMS = {
        'T': 180,           # 🔥 固定T=180
        'Cmax': 5,
        'window_size': 10,
        'advanced': 3,
        'K': 5.993,
        'Calpha': 0.01,
        'Cbeta': 0.02,
        'Cs': 100
    }
    
    # 🔥 🔥 🔥 N敏感性分析范围
    N_values = [20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]  # 测试的N值
    # task_densities = [3, 6, 13]  # 密度范围
    task_densities = [1, 12]  # 密度范围
    seeds = list(range(1, 2))  # 10个seeds
    
    print("🚀 简单聚合策略模型 - 开始N敏感性分析（固定T=180）")
    print(f"N范围: {N_values}")
    print(f"密度: {task_densities}")
    print(f"每个组合: {len(seeds)} seeds")
    print(f"总实验数: {len(task_densities)*len(N_values)*len(seeds)}")
    
    # # 🔥 🔥 🔥 运行完整分析
    # grand_summary = run_N_sensitivity_analysis_simple(
    #     base_params=BASE_PARAMS,
    #     task_densities=task_densities,
    #     N_values=N_values,
    #     seeds=seeds,
    #     output_dir='N_sensitivity_3Days_results_2026'
    # )
    # 🔥 🔥 🔥 运行完整分析
    grand_summary = run_N_sensitivity_analysis_simple(
        base_params=BASE_PARAMS,
        task_densities=task_densities,
        N_values=N_values,
        seeds=seeds,
        output_dir='N_3Days_results_2026'
    )
    
    print("\n✅ 简单聚合策略所有分析完成！")
    print("\n📁 输出文件结构：")
    print("   N_3Days_results_2026/")
    print("   ├── density_3_N_20_all_seeds.csv")
    print("   ├── density_3_N_30_all_seeds.csv")
    print("   ├── ... (3密度×10N=30个详细CSV)")
    print("   ├── SUMMARY_density_3_all_N.csv")
    print("   └── GRAND_SUMMARY_all_densities_all_N.csv")
    
    print("\n📊 大汇总预览：")
    print(grand_summary.round(3).head(10))


🚀 简单聚合策略模型 - 开始N敏感性分析（固定T=180）
N范围: [20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
密度: [1, 12]
每个组合: 1 seeds
总实验数: 22

🔍 Simple Strategy - 任务密度: 1 tasks/day

  📊 Testing N=20...
     🌱 Seed 1...
Generated 15 tasks (target_daily_tasks=1.0)

===== 从数据中提取维修周期 =====
Caisson    间隔数        间隔范围                 周期           D_min      D_max     
--------------------------------------------------------------------------------
0          ... 10.00        ...
1          ... 10.00        ...
2          ... 10.00        ...
3          ... 10.00        ...
4          ... 10.00        ...
5          ... 10.00        ...
8          ... 10.00        ...
9          ... 10.00        ...
11         ... 10.00        ...
13         ... 10.00        ...
14         ... 10.00        ...
16         ... 10.00        ...
17         ... 10.00        ...
18         ... 10.00        ...
19         ... 10.00        ...
===== 提取完成 =====

Window 0: N_tasks=3
  → 聚合完成: 2 个聚类点
Window 10: N_tasks=1
  → 聚合完成: 1 个聚类点
Wind

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

     ✅ N=60 results saved: N_3Days_results_2026/density_1_N_60_all_seeds.csv

  📊 Testing N=70...
     🌱 Seed 1...
Generated 63 tasks (target_daily_tasks=1.0)

===== 从数据中提取维修周期 =====
Caisson    间隔数        间隔范围                 周期           D_min      D_max     
--------------------------------------------------------------------------------
0          ... 10.00        ...
3          ... 10.00        ...
4          ... 10.00        ...
5          ... 10.00        ...
6          ... 10.00        ...
8          ... 10.00        ...
9          1          [152, 152]           152.00       92         213       
10         ... 10.00        ...
11         ... 10.00        ...
12         ... 10.00        ...
13         ... 10.00        ...
14         ... 10.00        ...
15         ... 10.00        ...
16         ... 10.00        ...
19         ... 10.00        ...
20         1          [154, 154]           154.00       93         216       
21         ... 10.00        ...
22         ... 10.00  

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

  → 聚合完成: 2 个聚类点
     ✅ N=90 results saved: N_3Days_results_2026/density_1_N_90_all_seeds.csv

  📊 Testing N=100...
     🌱 Seed 1...
Generated 88 tasks (target_daily_tasks=1.0)

===== 从数据中提取维修周期 =====
Caisson    间隔数        间隔范围                 周期           D_min      D_max     
--------------------------------------------------------------------------------
0          ... 10.00        ...
1          ... 10.00        ...
2          ... 10.00        ...
3          ... 10.00        ...
4          ... 10.00        ...
5          ... 10.00        ...
6          ... 10.00        ...
7          ... 10.00        ...
8          ... 10.00        ...
9          ... 10.00        ...
10         ... 10.00        ...
11         ... 10.00        ...
13         ... 10.00        ...
14         ... 10.00        ...
15         ... 10.00        ...
16         ... 10.00        ...
17         ... 10.00        ...
18         ... 10.00        ...
19         1          [166, 166]           166.00       100     

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

... (32 outputs left out)

Window 60: N_tasks=62
  → 聚合完成: 11 个聚类点
Window 70: N_tasks=47
  → 聚合完成: 11 个聚类点
Window 80: N_tasks=43
  → 聚合完成: 10 个聚类点
Window 90: N_tasks=51
  → 聚合完成: 10 个聚类点
Window 100: N_tasks=53
  → 聚合完成: 11 个聚类点
Window 110: N_tasks=48
  → 聚合完成: 10 个聚类点
Window 120: N_tasks=45
  → 聚合完成: 11 个聚类点


/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

Window 130: N_tasks=60
  → 聚合完成: 11 个聚类点
Window 140: N_tasks=43
  → 聚合完成: 11 个聚类点
Window 150: N_tasks=51
  → 聚合完成: 12 个聚类点
Window 160: N_tasks=57
  → 聚合完成: 10 个聚类点
Window 170: N_tasks=30
  → 聚合完成: 8 个聚类点
     ✅ N=120 results saved: N_3Days_results_2026/density_12_N_120_all_seeds.csv
  📁 Density 12 summary: N_3Days_results_2026/SUMMARY_density_12_all_N.csv

🎉 Simple Strategy总汇总保存: N_3Days_results_2026/GRAND_SUMMARY_all_densities_all_N.csv

✅ 简单聚合策略所有分析完成！

📁 输出文件结构：
   N_3Days_results_2026/
   ├── density_3_N_20_all_seeds.csv
   ├── density_3_N_30_all_seeds.csv
   ├── ... (3密度×10N=30个详细CSV)
   ├── SUMMARY_density_3_all_N.csv
   └── GRAND_SUMMARY_all_densities_all_N.csv

📊 大汇总预览：
   seed    N  target_daily_tasks  nb_groups  Avg_tasks/group  CS_cost_before  \
0     1   20                   1         14            1.071            1500   
1     1   30                   1         25            1.040            2600   
2     1   40                   1         30            1.300            3

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_59395/3156164497.py:261: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals